In [ ]:
import math

import matplotlib.pyplot as plt
import xarray as xr

# Description

Notebook to compare new and current reference ("gold") NetCDF files used to verify outputs of the SMRF test suite.

This notebook creates spatial difference plots between two NetCDF files, subtracting the "old" from the "new". This comparison is required whenever there is a need to update the "gold" references files due to test failures.

## Setup
### Change the below to the SMRF test `basins` folder location

In [ ]:
%cd /data/projects/iSnobal-CM/smrf/smrf/tests/basins/

### Test basin

In [ ]:
basin_name = "Lakes"

### NetCDF filename and variable to be checked

In [ ]:
filename = "wind_direction"
variable_name = "wind_direction"

### Path to the NetCDF files

In [ ]:
old_nc = xr.open_dataset(f"{basin_name}/gold_hrrr/{filename}.nc").drop_vars("projection")
new_nc = xr.open_dataset(f"{basin_name}/output/{filename}.nc").drop_vars("projection")

## Comparison

In [ ]:
def plot_file(file, variable):
    plots = file[variable].plot(
        x='x', y='y', col="time",
        figsize=(11, 5),
        subplot_kws={"box_aspect": 1},
        cbar_kwargs={"shrink": 0.5}
    )
    plots.set_titles("{value}")
    for ax, meta in zip(plots.axs.flat, plots.name_dicts.flat):
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    return plots

### New File

In [ ]:
plot_file(new_nc, variable_name);

### Current File

In [ ]:
plot_file(old_nc, variable_name);

### Difference

In [ ]:
diff = new_nc - old_nc

In [ ]:
plots = plot_file(diff, variable_name)

In [ ]:
old_nc.close()
new_nc.close()

In [ ]:
plots.fig.savefig(f"notebooks/{filename}_{variable_name}_xy.png", dpi=120)

### Histograms

In [ ]:
num_times = diff.sizes["time"]
num_rows = math.ceil(num_times / 2)

In [ ]:
fig, axes = plt.subplots(num_rows, 2, figsize=(10, 4 * num_rows), dpi=120)
axes = axes.flatten()

for i in range(num_times):
    diff[variable_name].isel(time=i).plot.hist(ax=axes[i], bins=30, color='skyblue')

fig.tight_layout()

In [ ]:
fig.savefig(f"notebooks/{filename}_{variable_name}_hist.png", dpi=120)